# 03 模拟 A/B 实验与统计检验

> **重要声明：本 notebook 的 A/B 实验是模拟数据，仅用于演示统计检验方法，不是 Olist 真实实验结果。**
>
> Olist 数据集没有真实的曝光、分组、营销成本字段，因此无法评估真实营销增量。此处用代码模拟一次随机分组实验，练习：
> - 比例差 z 检验（两独立样本比例检验）
> - 95% 置信区间
> - 近似 MDE（Minimum Detectable Effect）计算
>
> 随机种子固定为 `20260724`，结果可复现。

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

SEED = 20260724
DATA_DIR = Path('..') / 'dashboard' / 'data'

# 读取已生成的模拟结果
with open(DATA_DIR / 'ab_simulation.json', 'r', encoding='utf-8') as f:
    ab_result = json.load(f)

print(f"实验类型: {ab_result['type']}")
print(f"对照组: {ab_result['control_n']:,} 人，转化率 {ab_result['control_rate']:.4f}")
print(f"实验组: {ab_result['treatment_n']:,} 人，转化率 {ab_result['treatment_rate']:.4f}")
print(f"提升量: {ab_result['uplift_pp']*100:.2f}pp")
print(f"95% CI: [{ab_result['ci95_low']*100:.2f}pp, {ab_result['ci95_high']*100:.2f}pp]")
print(f"z = {ab_result['z_stat']:.4f}, p = {ab_result['p_value']:.2e}")
print(f"近似 MDE (80% power): {ab_result['approx_mde_pp_80_power']*100:.2f}pp")

## 1. 模拟实验设计

**模拟逻辑（与 `src/analysis.py` 中 `ab_simulation` 一致）：**
1. 从有效用户中随机分为对照组/实验组（各约 50%）
2. 基础转化率设为 ~10%（二项分布）
3. 实验组额外施加 ~1.8% 的转化提升（模拟营销效果）
4. 最终转化 = max(基础转化, 实验组提升)

这是一个**人为构造**的实验，目的是演示统计方法，不是真实业务效果。

In [ ]:
# 复现模拟过程
rng = np.random.default_rng(SEED)

# 模拟用户池（用 summary 中的用户数）
with open(DATA_DIR / 'summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)
n_users = summary['valid_users']

users = pd.DataFrame({'user_id': range(n_users)})
users['group'] = rng.choice(['control', 'treatment'], n_users)

# 基础转化 ~10%
base = rng.binomial(1, 0.100, n_users)
# 实验组额外提升 ~1.8%
effect = (users['group'].eq('treatment') & (rng.random(n_users) < 0.018)).astype(int)
users['converted'] = np.maximum(base, effect)

control = users[users['group'] == 'control']['converted']
treatment = users[users['group'] == 'treatment']['converted']

print(f"对照组: n={len(control)}, 转化率={control.mean():.4f}")
print(f"实验组: n={len(treatment)}, 转化率={treatment.mean():.4f}")
print(f"实际提升: {(treatment.mean() - control.mean())*100:.2f}pp")

## 2. 比例差 z 检验

**检验假设：**
- H₀：两组转化率无差异（p₁ = p₂）
- H₁：两组转化率有差异（p₁ ≠ p₂）

**检验统计量：**
$$z = \frac{\hat{p}_1 - \hat{p}_2}{\sqrt{\hat{p}_1(1-\hat{p}_1)/n_1 + \hat{p}_2(1-\hat{p}_2)/n_2}}$$

在 H₀ 下，z 近似服从标准正态分布 N(0,1)。

In [ ]:
# 手动计算 z 检验（不直接调 scipy，展示公式）
p1 = treatment.mean()
p2 = control.mean()
n1 = len(treatment)
n2 = len(control)

diff = p1 - p2
se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
z = diff / se
p_value = 2 * stats.norm.sf(abs(z))  # 双侧检验

print(f"=== 比例差 z 检验 ===")
print(f"转化率差: {diff*100:.4f}pp")
print(f"标准误 SE: {se:.6f}")
print(f"z 统计量: {z:.4f}")
print(f"p 值: {p_value:.2e}")
print(f"α = 0.05，{'拒绝 H₀，两组差异显著' if p_value < 0.05 else '不能拒绝 H₀'}")

# 与 scipy 结果对比
z_scipy, p_scipy = stats.proportions_ztest(
    [treatment.sum(), control.sum()],
    [n1, n2],
    alternative='two-sided'
)
print(f"\nscipy 验证: z={z_scipy:.4f}, p={p_scipy:.2e}")

## 3. 置信区间

95% 置信区间：
$$\text{CI} = (\hat{p}_1 - \hat{p}_2) \pm 1.96 \times SE$$

CI 不包含 0 即表示在 α=0.05 水平下显著。

In [ ]:
ci_low = diff - 1.96 * se
ci_high = diff + 1.96 * se

print(f"95% 置信区间: [{ci_low*100:.2f}pp, {ci_high*100:.2f}pp]")
print(f"CI 包含 0: {'是（不显著）' if ci_low <= 0 <= ci_high else '否（显著）'}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(diff*100, 0, xerr=[[(diff-ci_low)*100], [(ci_high-diff)*100]],
            fmt='o', color='steelblue', capsize=10, markersize=10, linewidth=2)
ax.axvline(0, color='red', linestyle='--', label='零差异线')
ax.set_xlabel('转化率提升 (pp)')
ax.set_title('A/B 实验转化率差及 95% 置信区间')
ax.legend()
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 4. MDE（最小可检测效应）

在给定样本量、α、power 下，实验能检测到的最小效应量。

**近似公式（两组等规模）：**
$$MDE = (z_{1-\alpha/2} + z_{1-\beta}) \times \sqrt{\frac{2\bar{p}(1-\bar{p})}{n_{harm}}}$$

其中 $\bar{p}$ 为混合转化率，$n_{harm}$ 为两组调和平均样本量。

In [ ]:
alpha = 0.05
power = 0.80
pooled_p = (treatment.sum() + control.sum()) / (n1 + n2)
n_harm = 2 / (1/n1 + 1/n2)

z_alpha = stats.norm.ppf(1 - alpha/2)  # 1.96
z_beta = stats.norm.ppf(power)         # 0.8416

mde = (z_alpha + z_beta) * np.sqrt(2 * pooled_p * (1 - pooled_p) / n_harm)

print(f"=== MDE 计算 ===")
print(f"混合转化率 p̅: {pooled_p:.4f}")
print(f"调和平均样本量: {n_harm:,.0f}")
print(f"z_(1-α/2) = {z_alpha:.4f}")
print(f"z_(1-β) = {z_beta:.4f}")
print(f"近似 MDE: {mde*100:.2f}pp (双侧 α=0.05, power=0.80)")
print()
print(f"实际提升 {diff*100:.2f}pp > MDE {mde*100:.2f}pp，实验有足够检出力。")

## 5. 结果解读与局限

**模拟结果：**
- 实验组转化率比对照组高 1.91pp，95% CI [1.51pp, 2.31pp]，p < 0.001
- MDE 为 0.57pp，实际提升远超检出阈值

**但必须强调：**
1. **这是模拟数据**——提升效果是代码人为注入的，不是 Olist 真实营销效果
2. **真实实验需要**：预注册主指标、按用户随机分组、采集曝光与成本数据、以增量 GMV 或增量利润判定
3. **不能直接外推**：模拟的转化率基数（10%）和提升量（1.8%）都是假设值
4. **本项目的 A/B 部分仅用于展示统计方法掌握程度**，不作为业务结论

**真实业务中做 A/B 实验的 checklist：**
- [ ] 实验前确定主指标、护栏指标、样本量/MDE
- [ ] 按用户（而非订单）随机分组，避免污染
- [ ] 记录曝光、点击、转化、成本全链路
- [ ] 实验结束后做显著性检验 + 置信区间 + 增量收益测算
- [ ] 扣除成本后以增量利润判定，而非仅看转化率